In [1]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df[["time", "X", "Y", "ndvi"]]

In [ ]:
import xee
help(xee.EarthEngineBackendEntrypoint)

In [5]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()

True

In [1]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
WSDemo_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")
Park_Lane_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/park_lane_tahoe")
#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-05-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=Park_Lane_Boundaries.geometry(), crs='EPSG:3310', scale=30)

In [ ]:
print(ds)

In [ ]:
# 5. Preview the dataset and the results before doing a full run!
# This allows users to inspect the structure and content of the data to ensure it behaves as expected prior to running a full computation.
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30
},
    user_function=compute_ndvi,
    preview_dataset=True
)

In [ ]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df = pd.DataFrame(columns=["X", "Y", "SR_B4", "SR_B5", "ndvi"])
df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    tune_function = False,
    max_pixels_per_tile = 1_000_000,
    dataset_config={
        'geometry': US_Boundaries,
        'crs': 'EPSG:5070',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        "chunks": chunks,
        "max_iterations": None,
        "output_template": df_list
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "US_NDVI_Tiles_30m_1MIL_nodocker",
        "vrt": True,
        "report": True
    },
        #"target_blocks_per_run": 2000},
    dask_mode="custom",
    dask_config={
        "n_workers": 6,
        "threads_per_worker": 1,
        "memory_limit": "3g",
        #"ee_max_concurrency": 30
    },
    #dask_use_docker=True,
    #dask_docker_image="adrianomdocker/rr042"
)
#[robustraster] ❌ ERROR during run(): Too Many Requests: Request was rejected because the request rate or concurrency limit was exceeded.
# EEException

In [ ]:
# Docker version
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    tune_function = False,
    max_pixels_per_tile = 1_000_000,
    dataset_config={
        'geometry': US_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        "chunks": chunks,
        "max_iterations": None,
        "output_template": df_list
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "US_NDVI_Tiles_30m_10mil_docker",
        "vrt": True,
        "report": True
    },
        #"target_blocks_per_run": 2000},
    dask_mode="custom",
    dask_config={
        "n_workers": 4,
        "threads_per_worker": 1,
        "memory_limit": "3g",
        #"ee_max_concurrency": 30
    },
    dask_use_docker=True,
    dask_docker_image="adrianomdocker/rr042r"
)

### Using R code instead of Python

You can also run your function via R code on the Docker workers! Instead of a Python function, you define your function as an R script string.

In [3]:
r_code = """\
compute_ndvi_r <- function(df) {
    df$ndvi <- (df$SR_B5 - df$SR_B4) / (df$SR_B5 + df$SR_B4)
    return(df[, c("time", "X", "Y", "ndvi")])
}
"""

df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}
from robustraster import run

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    tune_function = False,
    dataset_config={
        'geometry': Park_Lane_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "is_r_function": True,
        "r_function_code": r_code,
        "r_function_name": "compute_ndvi_r",
    },
    function_tuning_config={
        "chunks": chunks,
        "max_iterations": None,
        "output_template": df_list
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "PL_Tuned_NDVI_Tiles_30m_40MIL_R",
        "vrt": True,
        "report": True
    },
    dask_mode="custom",
    dask_config={
        "n_workers": 4,
        "threads_per_worker": 1,
        "memory_limit": "3g",
    },
    dask_use_docker=True,
    dask_docker_image="adrianomdocker/r042"
)

[robustraster] Found EE credentials at: C:\Users\Adriano Matos/.config/earthengine/credentials
[robustraster] Mounting EE credentials to /root/.config/earthengine/credentials
[robustraster] Mounting output: C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\demos\PL_Tuned_NDVI_Tiles_30m_40MIL_R -> /robustraster_output
Dask Scheduler started: dask-e7f4c4-scheduler (5c547298c06f)
Dask Worker 1 started: dask-e7f4c4-worker-1 (a037d017bb1a) mapped to port 57187
Dask Worker 2 started: dask-e7f4c4-worker-2 (830abb191e27) mapped to port 57188
Dask Worker 3 started: dask-e7f4c4-worker-3 (4258090a4a07) mapped to port 57189
Dask Worker 4 started: dask-e7f4c4-worker-4 (19c2dca891cb) mapped to port 57190


c:\anaconda3\envs\rr042_rpy2\lib\site-packages\distributed\client.py:1617: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.5.4  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Docker Dask cluster started: <Client: 'tcp://172.20.0.2:8786' processes=0 threads=0, memory=0 B>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] Running user function...
Exported: /robustraster_output/x-1153_-1093_y135250_135280__time_20180407T183853.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-1153_-1093_y135250_135280__time_20180101T183939.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-1153_-1093_y135250_135280__time_20180423T183844.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-1153_-1093_y135250_135280__time_20180218T183917.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-1153_-1093_y135250_135280__time_20180202T183922.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-1153_-1093_y135250_135280__time_20180509T183834.tif with bands [np.str_('ndvi')]
Exported: /robustraster_output/x-1153_-1093_

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\osgeo\gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


VRT file created successfully: PL_Tuned_NDVI_Tiles_30m_40MIL_R\time_20180101T183939.vrt
VRT file created successfully: PL_Tuned_NDVI_Tiles_30m_40MIL_R\time_20180117T183931.vrt
VRT file created successfully: PL_Tuned_NDVI_Tiles_30m_40MIL_R\time_20180202T183922.vrt
VRT file created successfully: PL_Tuned_NDVI_Tiles_30m_40MIL_R\time_20180218T183917.vrt
VRT file created successfully: PL_Tuned_NDVI_Tiles_30m_40MIL_R\time_20180306T183909.vrt
VRT file created successfully: PL_Tuned_NDVI_Tiles_30m_40MIL_R\time_20180407T183853.vrt
VRT file created successfully: PL_Tuned_NDVI_Tiles_30m_40MIL_R\time_20180423T183844.vrt
VRT file created successfully: PL_Tuned_NDVI_Tiles_30m_40MIL_R\time_20180509T183834.vrt
[robustraster] Dask client closed.
[robustraster] Dask cluster shut down.
